## Load test and train data from drive!!

In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

data_dir = '/content/gdrive/MyDrive/NLP Project/data/'

Mounted at /content/gdrive


In [2]:
import numpy as np
import pickle
# Ensure data_dir is set correctly
DATASET_TAG = "abc"  # "cefr6" or "abc"
data_dir = '/content/gdrive/MyDrive/NLP Project/data/'

if DATASET_TAG == "cefr6":
    x_train_path = data_dir + "X_train.txt"
    x_test_path = data_dir + "X_test.txt"
    y_train_path = data_dir + "y_train.txt"
    y_test_path = data_dir + "y_test.txt"
else:
    x_train_path = data_dir + "X_train_simplified.txt"
    x_test_path = data_dir + "X_test_simplified.txt"
    y_train_path = data_dir + "y_train_simplified.txt"
    y_test_path = data_dir + "y_test_simplified.txt"

# Load X_train (sentences) as text
with open(x_train_path, 'r') as f:
    X_train = f.read().splitlines()

# Load X_test (sentences) as text
with open(x_test_path, 'r') as f:
    X_test = f.read().splitlines()

# Load y_train (labels) as text
with open(y_train_path, 'r') as f:
    y_train = [line.strip() for line in f.read().splitlines()]

# Load y_test (labels) as text
with open(y_test_path, 'r') as f:
    y_test = [line.strip() for line in f.read().splitlines()]

print(f"Loaded X_train samples: {len(X_train)}")
print(f"Loaded X_test samples: {len(X_test)}")
print(f"Loaded y_train samples: {len(y_train)}")
print(f"Loaded y_test samples: {len(y_test)}")

Loaded X_train samples: 15629
Loaded X_test samples: 3766
Loaded y_train samples: 15629
Loaded y_test samples: 3766


## Constituency Parser Baseline

Extracts syntactic complexity features (tree depth, subordination, phrase length, etc.)
from constituency parse trees using `benepar` + `spaCy`, then trains a logistic
regression classifier as a feature-engineered baseline to compare against the BERT approach.

In [3]:
#installs, don't run this again
!pip install benepar spacy
!python -m spacy download en_core_web_md
!pip install transformers==4.36.0

  Preparing metadata (setup.py) ... done
  Created wheel for benepar: filename=benepar-0.2.0-py3-none-any.whl size=37625 sha256=7e349095bca85e1c2dcabf76261e397577d30742db2f2d21865e89f5949a934b
  Stored in directory: /root/.cache/pip/wheels/9b/84/c1/f2ac877f519e2864e7dfe52a1c17fe5cdd50819cb8d1f1945f
Successfully built benepar
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 82.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━

In [4]:
import numpy as np
import spacy
import benepar
from nltk import Tree
import nltk
nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

Chose to copy from this: https://github.com/nikitakit/self-attentive-parser

In [5]:
nlp = spacy.load("en_core_web_md")
benepar.download("benepar_en3")
nlp.add_pipe("benepar", config={"model": "benepar_en3"})

[nltk_data] Downloading package benepar_en3 to /root/nltk_data...
[nltk_data]   Unzipping models/benepar_en3.zip.
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thouroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils

In [6]:
import os

features = [
    "tree_height", "sbar_count", "avg_phrase_len", "clause_count",
    "np_count", "coord_count", "subordination_ratio",
    # new 4
    "avg_token_depth", "avg_np_len", "ttr", "avg_word_len"]

def extract_features(sentence):
    try:
        doc = nlp(sentence)
        sent = list(doc.sents)[0]
        tree = Tree.fromstring(sent._.parse_string)
    except Exception:
        return [0.0] * len(features)

    phrase_lengths, np_lengths, token_depths = [], [], []
    sbar_count = np_count = coord_count = clause_count = 0

    # Traverse all nodes in the parse tree and count syntactic features
    for subtree in tree.subtrees():
        #Track the length of each phrase type
        label = subtree.label()
        leaves = len(subtree.leaves())
        if label in ("NP", "VP", "PP", "ADJP", "ADVP", "SBAR", "S"): #Noun Phrase, Verb Phrase, Prepositional Phrase, Adjacent Prhase, Adverb Phrase, Subordinate Clause, Sentence/clause
            phrase_lengths.append(leaves)
        if label == "SBAR":
          sbar_count += 1
        if label == "NP":
          np_count += 1
          np_lengths.append(leaves)
        if label == "CC":
          coord_count += 1
        if label in ("S", "SBAR"):
          clause_count += 1
        if leaves  == 1:
          token_depths.append(len(subtree.treepositions()))

    tokens = [t.text.lower() for t in doc if t.is_alpha]

    avg_phrase_len = 0.0
    avg_np_len = 0.0
    avg_token_depth = 0.0
    ttr = 0.0
    avg_word_len = 0.0
    subordination_ratio = 0.0


    if phrase_lengths:
        avg_phrase_len = float(np.mean(phrase_lengths))
    if np_lengths:
        avg_np_len = float(np.mean(np_lengths))
    if token_depths:
        avg_token_depth = float(np.mean(token_depths))
    if tokens:
        ttr = len(set(tokens)) / len(tokens)
        avg_word_len = float(np.mean([len(t) for t in tokens]))
    if clause_count > 0:
        subordination_ratio = sbar_count / clause_count

    return [tree.height(), sbar_count, avg_phrase_len, clause_count,
        np_count, coord_count, subordination_ratio,
        avg_token_depth, avg_np_len, ttr, avg_word_len] #kinda self explanitory
if DATASET_TAG == "cefr6":
    train_feats_path = data_dir + "X_train_feats_og.npy"
    test_feats_path  = data_dir + "X_test_feats_og.npy"
else:
    train_feats_path = data_dir + "X_train_feats_abc.npy"
    test_feats_path  = data_dir + "X_test_feats_abc.npy"

#Takes a lot of time so I want to cache it so that we can fast access it again
if os.path.exists(train_feats_path) and os.path.exists(test_feats_path):
    X_train_feats = np.load(train_feats_path)
    X_test_feats  = np.load(test_feats_path)
    print(f"Loaded cached features. Shape: {X_train_feats.shape}")
else:
    X_train_feats = []
    for i, s in enumerate(X_train):
        X_train_feats.append(extract_features(s))
        if i % 100 == 0:
            print(f"{i}/{len(X_train)}")
    X_train_feats = np.array(X_train_feats)
    X_test_feats  = np.array([extract_features(s) for s in X_test])
    np.save(train_feats_path, X_train_feats)
    np.save(test_feats_path, X_test_feats)
    print("Features saved to Drive.")

Loaded cached features. Shape: (15629, 11)


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_feats)
X_test_scaled  = scaler.transform(X_test_feats)

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train_scaled, y_train)

y_pred = clf.predict(X_test_scaled)

if DATASET_TAG == "cefr6":
    label_names = ["A1", "A2", "B1", "B2", "C1", "C2"]
else:
    label_names = ["A", "B", "C"]

print("----- Constituency Parser Baseline -----")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print(classification_report(y_test, y_pred, target_names=label_names, digits = 4))

----- Constituency Parser Baseline -----
Accuracy: 0.7379

              precision    recall  f1-score   support

           A     0.8345    0.8755    0.8545      1446
           B     0.6299    0.6632    0.6461      1247
           C     0.7329    0.6393    0.6829      1073

    accuracy                         0.7379      3766
   macro avg     0.7324    0.7260    0.7279      3766
weighted avg     0.7378    0.7379    0.7366      3766

